# Transformer Training

Train a decoder-only Transformer on labelled candle feature pickle files, save a checkpoint, then reload it for a quick inference check.


## Imports / PyTorch Load

Run this first so the slow one-time PyTorch and CUDA initialization happens before changing settings.


In [21]:
from datetime import datetime
from pathlib import Path
import json
import os
import random
import sys
import time

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PACKAGE_SRC = PROJECT_ROOT / "packages" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

from tradingbot.models import TransformerWindowDataset, build_transformer

{
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}


{'torch_version': '2.7.1+cu118',
 'cuda_available': True,
 'cuda_version': '11.8',
 'cuda_device': 'GeForce GTX 1060'}

## Settings

Configure paths, dataset filters, feature selection, model hyperparameters, and runtime settings here.


In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "historic_candle_feature"
MODEL_DIR = PROJECT_ROOT / "data" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FILE_GLOB = "*_features_triple_barrier.pkl"
TIMEFRAME_FILE_PATTERN = "*_60minute_*_features_triple_barrier.pkl"  # set None to use every labelled file
PICKLE_COMPRESSION = "gzip"
CHECKPOINT_PREFIX = "triple_barrier_transformer"
SEED = 42

OUTPUT_TYPE = "classification"  # "classification" or "regression"
CONTEXT_LENGTH = 128
TARGET_OFFSET = 0  # triple-barrier labels are already aligned to the candle being predicted
BATCH_SIZE = 512
EPOCHS = 1000
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
MAX_ROWS_PER_FILE = None  # set None for full training
MAX_FILES = 5  # set None to use all matched files
TRAIN_SPLIT = 0.8

DATALOADER_WORKERS = 0  # fastest for this preloaded in-memory dataset on Windows/Jupyter
PREFETCH_FACTOR = 2
USE_AMP = False  # GTX 1060 has no Tensor Cores; AMP was slower in local benchmarks
EVAL_EVERY_EPOCHS = 5
EVALUATE_TRAIN_METRICS = False  # validation is enough for checkpointing and saves an extra full train pass
SAVE_EVERY_EPOCHS = 10

MODEL_DIM = 32
NUM_HEADS = 4
NUM_LAYERS = 4
DROPOUT = 0.1

LABEL_COLUMNS = [
    "triple_barrier_label_lower",
    "triple_barrier_label_neutral",
    "triple_barrier_label_upper",
]
REGRESSION_COLUMNS = ["triple_barrier_return"]

SELECTED_FEATURE_COLUMNS = [
    "upper_wick_to_candle_ratio",
    "lower_wick_to_candle_ratio",
    "body_to_candle_ratio",
    "candle_type_standard",
    "candle_type_doji",
    "candle_type_hammer",
    "candle_type_inverted_hammer",
    "candle_type_spinning_top",
    "candle_type_marubozu",
    "candle_color_green",
    "candle_color_red",
    "log_volume_zscore",
    "rsi_7",
    "rsi_14",
    "stochastic_7",
    "stochastic_14",
    "stochastic_rsi_7",
    "stochastic_rsi_14",
    "macd",
    "atr",
    "adx",
]
# SELECTED_FEATURE_COLUMNS = None  # uncomment to use every available feature column

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = USE_AMP and DEVICE.type == "cuda"

{
    "project_root": PROJECT_ROOT,
    "device": DEVICE,
    "output_type": OUTPUT_TYPE,
    "file_pattern": TIMEFRAME_FILE_PATTERN or FILE_GLOB,
    "amp_enabled": USE_AMP,
    "dataloader_workers": DATALOADER_WORKERS,
    "evaluate_train_metrics": EVALUATE_TRAIN_METRICS,
}


{'project_root': WindowsPath('d:/PROJECTS/MY PROJECTS/AlphaBot/alphabot'),
 'device': device(type='cuda'),
 'output_type': 'classification',
 'file_pattern': '*_60minute_*_features_triple_barrier.pkl',
 'amp_enabled': False,
 'dataloader_workers': 0,
 'evaluate_train_metrics': False}

## Dataset Loading

Match labelled pickle files, infer the schema, validate selected features and target columns, split files, then build datasets and dataloaders.


In [23]:
labelled_files = sorted(DATA_DIR.glob(FILE_GLOB))
if TIMEFRAME_FILE_PATTERN is not None:
    labelled_files = sorted(path for path in labelled_files if path.match(TIMEFRAME_FILE_PATTERN))
if not labelled_files:
    pattern = TIMEFRAME_FILE_PATTERN or FILE_GLOB
    raise FileNotFoundError(f"No labelled feature files found at {DATA_DIR} matching {pattern}")

schema = pd.read_pickle(labelled_files[0], compression=PICKLE_COMPRESSION).columns.tolist()
excluded_columns = {"timestamp", *LABEL_COLUMNS, *REGRESSION_COLUMNS}
available_feature_columns = [column for column in schema if column not in excluded_columns]
feature_columns = available_feature_columns if SELECTED_FEATURE_COLUMNS is None else list(SELECTED_FEATURE_COLUMNS)
unknown_features = [column for column in feature_columns if column not in available_feature_columns]
if unknown_features:
    raise ValueError(f"Selected features are not present in the dataset: {unknown_features}")
if not feature_columns:
    raise ValueError("Select at least one feature column.")

if OUTPUT_TYPE == "classification":
    missing_targets = [column for column in LABEL_COLUMNS if column not in schema]
    target_mode = "class_index"
    target_kwargs = {"label_columns": LABEL_COLUMNS}
    output_dim = len(LABEL_COLUMNS)
else:
    missing_targets = [column for column in REGRESSION_COLUMNS if column not in schema]
    target_mode = "regression"
    target_kwargs = {"target_columns": REGRESSION_COLUMNS}
    output_dim = len(REGRESSION_COLUMNS)

if missing_targets:
    raise ValueError(f"Missing target columns in {labelled_files[0]}: {missing_targets}")

matched_files = [path.name for path in labelled_files]
training_files = labelled_files.copy()
if MAX_FILES is not None:
    training_files = training_files[:MAX_FILES]
random.shuffle(training_files)
split_index = max(1, int(len(training_files) * TRAIN_SPLIT))
train_files = training_files[:split_index]
val_files = training_files[split_index:] or training_files[:1]

train_dataset = TransformerWindowDataset(
    train_files,
    feature_columns=feature_columns,
    context_length=CONTEXT_LENGTH,
    target_mode=target_mode,
    target_offset=TARGET_OFFSET,
    max_rows_per_file=MAX_ROWS_PER_FILE,
    pickle_compression=PICKLE_COMPRESSION,
    **target_kwargs,
)
val_dataset = TransformerWindowDataset(
    val_files,
    feature_columns=feature_columns,
    context_length=CONTEXT_LENGTH,
    target_mode=target_mode,
    target_offset=TARGET_OFFSET,
    max_rows_per_file=MAX_ROWS_PER_FILE,
    pickle_compression=PICKLE_COMPRESSION,
    **target_kwargs,
)

if len(train_dataset) == 0:
    raise ValueError(f"Training dataset is empty. Skipped files: {train_dataset.skipped[:5]}")
if len(val_dataset) == 0:
    raise ValueError(f"Validation dataset is empty. Skipped files: {val_dataset.skipped[:5]}")

pin_memory = DEVICE.type == "cuda"
loader_kwargs = {
    "num_workers": DATALOADER_WORKERS,
    "pin_memory": pin_memory,
    "persistent_workers": DATALOADER_WORKERS > 0,
}
if DATALOADER_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

print(f"Matched {len(matched_files)} files using pattern: {TIMEFRAME_FILE_PATTERN or FILE_GLOB}")
{
    "loaded_files": len(training_files),
    "train_files": [path.name for path in train_files],
    "val_files": [path.name for path in val_files],
    "train_windows": len(train_dataset),
    "val_windows": len(val_dataset),
    "feature_dim": len(feature_columns),
    "feature_columns": feature_columns,
    "target_mode": target_mode,
    "train_skipped": train_dataset.skipped[:5],
    "val_skipped": val_dataset.skipped[:5],
    "label_counts": None if train_dataset.label_counts is None else train_dataset.label_counts.tolist(),
    "dataloader_workers": DATALOADER_WORKERS,
    "pin_memory": pin_memory,
    "prefetch_factor": PREFETCH_FACTOR if DATALOADER_WORKERS > 0 else None,
}


Matched 37 files using pattern: *_60minute_*_features_triple_barrier.pkl


{'loaded_files': 5,
 'train_files': ['BAJAJ-AUTO_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.pkl',
  'ASIANPAINT_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.pkl',
  'AXISBANK_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.pkl',
  'BAJAJFINSV_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.pkl'],
 'val_files': ['ADANIPORTS_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.pkl'],
 'train_windows': 77252,
 'val_windows': 19320,
 'feature_dim': 21,
 'feature_columns': ['upper_wick_to_candle_ratio',
  'lower_wick_to_candle_ratio',
  'body_to_candle_ratio',
  'candle_type_standard',
  'candle_type_doji',
  'candle_type_hammer',
  'candle_type_inverted_hammer',
  'candle_type_spinning_top',
  'candle_type_marubozu',
  'candle_color_green',
  'candle_color_red',
  'log_volume_zscore',
  'rsi_7',
  'rsi_14',
  'stochastic_7',
  'stochastic_14',
  'stochastic_rsi_7',
  'stochastic_rsi_14',
  'macd',


## Model Building

Create the Transformer, loss, class weights, and optimizer from the selected dataset shape and hyperparameters.


In [24]:
model_config = {
    "input_dim": len(feature_columns),
    "model_dim": MODEL_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "output_dim": output_dim,
    "output_type": OUTPUT_TYPE,
    "feature_columns": feature_columns,
    "file_glob": FILE_GLOB,
    "timeframe_file_pattern": TIMEFRAME_FILE_PATTERN,
    "pickle_compression": PICKLE_COMPRESSION,
    "matched_files": matched_files,
}
builder_config = {
    key: value
    for key, value in model_config.items()
    if key not in {"feature_columns", "file_glob", "timeframe_file_pattern", "pickle_compression", "matched_files"}
}
model = build_transformer(**builder_config).to(DEVICE)

class_weight = None
if OUTPUT_TYPE == "classification":
    if train_dataset.label_counts is not None and (train_dataset.label_counts > 0).all():
        counts = torch.tensor(train_dataset.label_counts, dtype=torch.float32, device=DEVICE)
        class_weight = counts.sum() / (counts.numel() * counts)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
else:
    criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

{
    "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    "model_config": model_config,
    "class_weight": None if class_weight is None else class_weight.detach().cpu().tolist(),
    "amp_enabled": USE_AMP,
}


{'parameter_count': 53891,
 'model_config': {'input_dim': 21,
  'model_dim': 32,
  'num_heads': 4,
  'num_layers': 4,
  'dropout': 0.1,
  'output_dim': 3,
  'output_type': 'classification',
  'feature_columns': ['upper_wick_to_candle_ratio',
   'lower_wick_to_candle_ratio',
   'body_to_candle_ratio',
   'candle_type_standard',
   'candle_type_doji',
   'candle_type_hammer',
   'candle_type_inverted_hammer',
   'candle_type_spinning_top',
   'candle_type_marubozu',
   'candle_color_green',
   'candle_color_red',
   'log_volume_zscore',
   'rsi_7',
   'rsi_14',
   'stochastic_7',
   'stochastic_14',
   'stochastic_rsi_7',
   'stochastic_rsi_14',
   'macd',
   'atr',
   'adx'],
  'file_glob': '*_features_triple_barrier.pkl',
  'timeframe_file_pattern': '*_60minute_*_features_triple_barrier.pkl',
  'pickle_compression': 'gzip',
  'matched_files': ['ADANIPORTS_60minute_1990-01-01T00-00-00plus05-30_now_features_triple_barrier.pkl',
   'ASIANPAINT_60minute_1990-01-01T00-00-00plus05-30_now_fea

## Training

Train for the configured number of epochs, print train and validation metrics, then save a checkpoint with the final validation accuracy in its name.


In [25]:
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = MODEL_DIR / f"{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_{RUN_TIMESTAMP}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_CSV_PATH = RUN_DIR / f"{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_{RUN_TIMESTAMP}_metrics.csv"
METRICS_JSONL_PATH = RUN_DIR / f"{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_{RUN_TIMESTAMP}_metrics.jsonl"


def move_batch(batch, device):
    x, y = batch
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)


def classification_metrics(correct, total, true_positive, false_positive, false_negative):
    precision = true_positive / (true_positive + false_positive).clamp_min(1)
    recall = true_positive / (true_positive + false_negative).clamp_min(1)
    f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-12)
    return {
        "accuracy": correct / max(total, 1),
        "precision_macro": precision.mean().item(),
        "recall_macro": recall.mean().item(),
        "f1_macro": f1.mean().item(),
        "precision_by_class": precision.cpu().tolist(),
        "recall_by_class": recall.cpu().tolist(),
        "f1_by_class": f1.cpu().tolist(),
    }


def checkpoint_metric(row):
    if OUTPUT_TYPE == "classification":
        metric_name = "val_accuracy"
        metric_value = row.get(metric_name)
        return metric_name, metric_value, metric_value if metric_value is not None else float("-inf")
    metric_name = "val_loss"
    metric_value = row.get(metric_name)
    return metric_name, metric_value, -metric_value if metric_value is not None else float("-inf")


def checkpoint_stem(epoch, tag, metric_name, metric_value):
    metric_text = "na" if metric_value is None else f"{metric_value:.4f}"
    return f"{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_{RUN_TIMESTAMP}_epoch_{epoch:04d}_{tag}_{metric_name}_{metric_text}"


def save_metrics(history):
    metrics_frame = pd.DataFrame(history)
    metrics_frame.to_csv(METRICS_CSV_PATH, index=False)
    return metrics_frame


def append_metrics_jsonl(row):
    with METRICS_JSONL_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, default=str) + "\n")


def save_training_artifacts(epoch, row, tag):
    metric_name, metric_value, _ = checkpoint_metric(row)
    stem = checkpoint_stem(epoch, tag, metric_name, metric_value)
    checkpoint_path = RUN_DIR / f"{stem}.checkpoint.pt"
    weights_path = RUN_DIR / f"{stem}.weights.pt"
    summary_path = RUN_DIR / f"{stem}.metrics.json"

    checkpoint_context = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "model_config": model_config,
        "feature_columns": feature_columns,
        "file_glob": FILE_GLOB,
        "timeframe_file_pattern": TIMEFRAME_FILE_PATTERN,
        "pickle_compression": PICKLE_COMPRESSION,
        "matched_files": matched_files,
        "train_files": [path.name for path in train_files],
        "val_files": [path.name for path in val_files],
        "label_columns": LABEL_COLUMNS if OUTPUT_TYPE == "classification" else None,
        "target_columns": REGRESSION_COLUMNS if OUTPUT_TYPE == "regression" else None,
        "context_length": CONTEXT_LENGTH,
        "target_offset": TARGET_OFFSET,
        "history": history,
        "metrics": row,
        "objective_type": OUTPUT_TYPE,
        "run_timestamp": RUN_TIMESTAMP,
        "amp_enabled": USE_AMP,
        "dataloader_workers": DATALOADER_WORKERS,
        "batch_size": BATCH_SIZE,
    }
    torch.save(checkpoint_context, checkpoint_path)
    torch.save(model.state_dict(), weights_path)

    summary = {
        "epoch": epoch,
        "tag": tag,
        "checkpoint_path": str(checkpoint_path),
        "weights_path": str(weights_path),
        "metrics_csv_path": str(METRICS_CSV_PATH),
        "metrics_jsonl_path": str(METRICS_JSONL_PATH),
        "metrics": row,
    }
    summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    return checkpoint_path, weights_path, summary_path


def prefixed_metrics(prefix, metrics):
    return {f"{prefix}_{key}": value for key, value in metrics.items()}


def train_one_epoch(model, loader, epoch):
    model.train()
    total_loss = 0.0
    total_items = 0
    progress = tqdm(
        loader,
        desc=f"Epoch {epoch:04d}/{EPOCHS} train",
        unit="batch",
        leave=False,
    )

    for batch in progress:
        x, y = move_batch(batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            prediction = model(x)
            loss = criterion(prediction, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        batch_size = x.size(0)
        total_loss += loss.detach().float().item() * batch_size
        total_items += batch_size
        progress.set_postfix(loss=f"{total_loss / max(total_items, 1):.4f}")

    return total_loss / max(total_items, 1)


@torch.no_grad()
def evaluate(model, loader, phase, epoch):
    model.eval()
    total_loss = 0.0
    total_items = 0
    progress = tqdm(
        loader,
        desc=f"Epoch {epoch:04d}/{EPOCHS} {phase}",
        unit="batch",
        leave=False,
    )

    if OUTPUT_TYPE == "classification":
        class_count = output_dim
        correct = 0
        true_positive = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)
        false_positive = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)
        false_negative = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)

    for batch in progress:
        x, y = move_batch(batch, DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            prediction = model(x)
            loss = criterion(prediction, y)

        batch_size = x.size(0)
        total_loss += loss.detach().float().item() * batch_size
        total_items += batch_size

        postfix = {"loss": f"{total_loss / max(total_items, 1):.4f}"}
        if OUTPUT_TYPE == "classification":
            predicted = prediction.argmax(dim=-1)
            correct += (predicted == y).sum().item()
            postfix["acc"] = f"{correct / max(total_items, 1):.4f}"
            for class_index in range(class_count):
                predicted_class = predicted == class_index
                actual_class = y == class_index
                true_positive[class_index] += (predicted_class & actual_class).sum()
                false_positive[class_index] += (predicted_class & ~actual_class).sum()
                false_negative[class_index] += (~predicted_class & actual_class).sum()
        progress.set_postfix(postfix)

    metrics = {"loss": total_loss / max(total_items, 1)}
    if OUTPUT_TYPE == "classification":
        metrics.update(
            classification_metrics(
                correct,
                total_items,
                true_positive,
                false_positive,
                false_negative,
            )
        )
    return metrics


history = []
best_score = float("-inf")
best_checkpoint_path = None
saved_checkpoint_path = None

print(f"Run directory: {RUN_DIR}")
print(f"Metrics CSV: {METRICS_CSV_PATH}")

for epoch in range(1, EPOCHS + 1):
    epoch_started_at = time.time()
    train_loss = train_one_epoch(model, train_loader, epoch)
    should_evaluate = epoch == 1 or epoch % EVAL_EVERY_EPOCHS == 0 or epoch == EPOCHS

    row = {
        "run_timestamp": RUN_TIMESTAMP,
        "epoch": epoch,
        "train_optimized_loss": train_loss,
        "elapsed_seconds": round(time.time() - epoch_started_at, 3),
        "evaluated": should_evaluate,
    }

    if should_evaluate:
        if EVALUATE_TRAIN_METRICS:
            train_metrics = evaluate(model, train_loader, "train_eval", epoch)
            row.update(prefixed_metrics("train", train_metrics))
        val_metrics = evaluate(model, val_loader, "val", epoch)
        row.update(prefixed_metrics("val", val_metrics))

    history.append(row)
    append_metrics_jsonl(row)
    metrics_frame = save_metrics(history)

    metric_name, metric_value, score = checkpoint_metric(row)
    checkpoint_reasons = []
    if epoch == 1 or epoch % SAVE_EVERY_EPOCHS == 0 or epoch == EPOCHS:
        checkpoint_reasons.append("periodic")
    if should_evaluate and score > best_score:
        best_score = score
        checkpoint_reasons.append("best")

    saved_paths = []
    for tag in dict.fromkeys(checkpoint_reasons):
        checkpoint_path, weights_path, summary_path = save_training_artifacts(epoch, row, tag)
        saved_checkpoint_path = checkpoint_path
        saved_paths.append({"tag": tag, "checkpoint": checkpoint_path.name, "weights": weights_path.name})
        if tag == "best":
            best_checkpoint_path = checkpoint_path

    print({**row, "checkpoint_metric": metric_name, "checkpoint_value": metric_value, "saved": saved_paths})

final_metrics = history[-1]
if saved_checkpoint_path is None:
    saved_checkpoint_path, _, _ = save_training_artifacts(EPOCHS, final_metrics, "final")

print(f"Saved latest checkpoint to {saved_checkpoint_path}")
if best_checkpoint_path is not None:
    print(f"Saved best checkpoint to {best_checkpoint_path}")

metrics_frame

Run directory: d:\PROJECTS\MY PROJECTS\AlphaBot\alphabot\data\models\triple_barrier_transformer_classification_20260526_012343
Metrics CSV: d:\PROJECTS\MY PROJECTS\AlphaBot\alphabot\data\models\triple_barrier_transformer_classification_20260526_012343\triple_barrier_transformer_classification_20260526_012343_metrics.csv


{'run_timestamp': '20260526_012343', 'epoch': 1, 'train_optimized_loss': 1.1055341049730207, 'elapsed_seconds': 399.601, 'evaluated': True, 'val_loss': 1.0771271692793316, 'val_accuracy': 0.5183229813664596, 'val_precision_macro': 0.33924350142478943, 'val_recall_macro': 0.3387339115142822, 'val_f1_macro': 0.3295849561691284, 'val_precision_by_class': [0.48409244418144226, 0.0, 0.5336380004882812], 'val_recall_by_class': [0.31793686747550964, 0.0, 0.6982648968696594], 'val_f1_by_class': [0.38380351662635803, 0.0, 0.6049513816833496], 'checkpoint_metric': 'val_accuracy', 'checkpoint_value': 0.5183229813664596, 'saved': [{'tag': 'periodic', 'checkpoint': 'triple_barrier_transformer_classification_20260526_012343_epoch_0001_periodic_val_accuracy_0.5183.checkpoint.pt', 'weights': 'triple_barrier_transformer_classification_20260526_012343_epoch_0001_periodic_val_accuracy_0.5183.weights.pt'}, {'tag': 'best', 'checkpoint': 'triple_barrier_transformer_classification_20260526_012343_epoch_0001_

,run_timestamp,epoch,train_optimized_loss,elapsed_seconds,evaluated,val_loss,val_accuracy,val_precision_macro,val_recall_macro,val_f1_macro,val_precision_by_class,val_recall_by_class,val_f1_by_class
0,20260526_012343,1,1.105534,399.601,True,1.077127,0.518323,0.339244,0.338734,0.329585,"[0.48409244418144226, 0.0, 0.5336380004882812]","[0.31793686747550964, 0.0, 0.6982648968696594]","[0.38380351662635803, 0.0, 0.6049513816833496]"


## Evaluation

Reload the latest checkpoint from this run and run a single validation-window inference check.


In [26]:
if "saved_checkpoint_path" in globals() and saved_checkpoint_path is not None:
    checkpoint_path = saved_checkpoint_path
else:
    checkpoint_candidates = sorted(
        MODEL_DIR.glob(f"**/{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_*.checkpoint.pt")
    )
    if not checkpoint_candidates:
        checkpoint_candidates = sorted(
            MODEL_DIR.glob(f"{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_*_valacc_*.pt")
        )
    checkpoint_path = checkpoint_candidates[-1]
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
checkpoint_model_config = checkpoint["model_config"]
checkpoint_builder_config = {
    key: value
    for key, value in checkpoint_model_config.items()
    if key not in {"feature_columns", "file_glob", "timeframe_file_pattern", "pickle_compression", "matched_files"}
}
inference_model = build_transformer(**checkpoint_builder_config).to(DEVICE)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model.eval()

x, y = val_dataset[0]
with torch.no_grad():
    output = inference_model(x.unsqueeze(0).to(DEVICE))

if checkpoint_model_config["output_type"] == "classification":
    probabilities = torch.softmax(output, dim=-1).squeeze(0).cpu()
    predicted_index = int(probabilities.argmax().item())
    inference_result = {
        "checkpoint": checkpoint_path.name,
        "features_used": checkpoint_model_config["feature_columns"],
        "timeframe_file_pattern": checkpoint_model_config.get("timeframe_file_pattern"),
        "predicted_label": checkpoint["label_columns"][predicted_index],
        "probabilities": dict(zip(checkpoint["label_columns"], probabilities.tolist())),
        "actual_index": int(y.item()),
        "actual_label": checkpoint["label_columns"][int(y.item())],
    }
else:
    prediction = output.squeeze(0).cpu().tolist()
    actual = y.cpu().tolist()
    inference_result = {
        "checkpoint": checkpoint_path.name,
        "features_used": checkpoint_model_config["feature_columns"],
        "timeframe_file_pattern": checkpoint_model_config.get("timeframe_file_pattern"),
        "prediction": dict(zip(checkpoint["target_columns"], prediction)),
        "actual": dict(zip(checkpoint["target_columns"], actual)),
    }

inference_result


{'checkpoint': 'triple_barrier_transformer_classification_20260526_012343_epoch_0001_best_val_accuracy_0.5183.checkpoint.pt',
 'features_used': ['upper_wick_to_candle_ratio',
  'lower_wick_to_candle_ratio',
  'body_to_candle_ratio',
  'candle_type_standard',
  'candle_type_doji',
  'candle_type_hammer',
  'candle_type_inverted_hammer',
  'candle_type_spinning_top',
  'candle_type_marubozu',
  'candle_color_green',
  'candle_color_red',
  'log_volume_zscore',
  'rsi_7',
  'rsi_14',
  'stochastic_7',
  'stochastic_14',
  'stochastic_rsi_7',
  'stochastic_rsi_14',
  'macd',
  'atr',
  'adx'],
 'timeframe_file_pattern': '*_60minute_*_features_triple_barrier.pkl',
 'predicted_label': 'triple_barrier_label_upper',
 'probabilities': {'triple_barrier_label_lower': 0.3594905436038971,
  'triple_barrier_label_neutral': 0.2801496684551239,
  'triple_barrier_label_upper': 0.36035972833633423},
 'actual_index': 2,
 'actual_label': 'triple_barrier_label_upper'}